In [31]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

import joblib

warnings.filterwarnings("ignore")

In [32]:
# Load dataset
DATA_PATH = r"C:\Users\DELL\Desktop\astro\data\processed\ALAZ_Engineered_Training_Data.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(32412, 58)


,SAT_NAME,NORAD_ID,EPOCH,DATE,BSTAR,MEAN_MOTION,PERIOD,KP_INDEX,ISN,F10.7_OBS,...,DOY_SIN,DOY_COS,HOUR_SIN,HOUR_COS,TARGET_NEXT_DELTA_BSTAR,TARGET_NEXT_DELTA_MEAN_MOTION,TARGET_BSTAR_UP,TARGET_MM_DROP,TARGET_ORBITAL_RESPONSE,NORAD_CAT_ID
0,TURKSAT 1B,23200,2025-01-02 03:33:23.222880000,2025-01-02,0.0,0.990462,2,20.0,187.0,212.4,...,0.034398,0.999408,0.707107,0.707107,0.0,4.430000e-06,0.0,0.0,0.0,23200
1,TURKSAT 1B,23200,2025-01-03 13:55:36.317280000,2025-01-03,0.0,0.990466,5,7.0,199.0,199.9,...,0.051584,0.998669,-0.258819,-0.965926,0.0,1.000000e-07,0.0,0.0,0.0,23200
2,TURKSAT 1B,23200,2025-01-03 15:43:07.616928000,2025-01-03,0.0,0.990466,6,13.0,199.0,199.9,...,0.051584,0.998669,-0.707107,-0.707107,0.0,3.740000e-06,0.0,0.0,0.0,23200
3,TURKSAT 1B,23200,2025-01-04 13:50:15.190080000,2025-01-04,0.0,0.990470,5,33.0,211.0,209.3,...,0.068755,0.997634,-0.258819,-0.965926,0.0,4.500000e-07,0.0,0.0,0.0,23200
4,TURKSAT 1B,23200,2025-01-04 15:31:44.751936000,2025-01-04,0.0,0.990471,6,43.0,211.0,209.3,...,0.068755,0.997634,-0.707107,-0.707107,0.0,2.920000e-06,0.0,0.0,0.0,23200


In [34]:
# Define target
TARGET = "TARGET_ORBITAL_RESPONSE"

# Drop non-feature columns
drop_cols = [
    "SAT_NAME",
    "EPOCH",
    "DATE",
    "TARGET_NEXT_DELTA_BSTAR",
    "TARGET_NEXT_DELTA_MEAN_MOTION",
    "TARGET_BSTAR_UP",
    "TARGET_MM_DROP",
    "TARGET_ORBITAL_RESPONSE"
]

X = df.drop(columns=drop_cols, errors="ignore")
y = df[TARGET]

print("Feature shape:", X.shape)
print("Target distribution:")
print(y.value_counts(normalize=True))

Feature shape: (32412, 50)
Target distribution:
TARGET_ORBITAL_RESPONSE
0.0    0.58392
1.0    0.41608
Name: proportion, dtype: float64


In [35]:
# Time-based split (CRITICAL)
df = df.sort_values("EPOCH")

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

X_train = train_df.drop(columns=drop_cols, errors="ignore")
y_train = train_df[TARGET]

X_test = test_df.drop(columns=drop_cols, errors="ignore")
y_test = test_df[TARGET]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (25929, 50)
Test : (6483, 50)


In [36]:
# Ensure no missing values remain in feature matrices
print("NaN count in X_train before cleanup:", X_train.isna().sum().sum())
print("NaN count in X_test before cleanup :", X_test.isna().sum().sum())

# Keep only rows with complete feature values
train_mask = X_train.notna().all(axis=1)
test_mask = X_test.notna().all(axis=1)

X_train = X_train.loc[train_mask].reset_index(drop=True)
y_train = y_train.loc[train_mask].reset_index(drop=True)

X_test = X_test.loc[test_mask].reset_index(drop=True)
y_test = y_test.loc[test_mask].reset_index(drop=True)

print("NaN count in X_train after cleanup:", X_train.isna().sum().sum())
print("NaN count in X_test after cleanup :", X_test.isna().sum().sum())

print("Cleaned train shape:", X_train.shape)
print("Cleaned test shape :", X_test.shape)

NaN count in X_train before cleanup: 154
NaN count in X_test before cleanup : 0
NaN count in X_train after cleanup: 0
NaN count in X_test after cleanup : 0
Cleaned train shape: (25807, 50)
Cleaned test shape : (6483, 50)


In [46]:
from sklearn.ensemble import ExtraTreesClassifier

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}

In [47]:
results = []

for name, model in models.items():
    print(f"\nTraining: {name}")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    roc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC-AUC": roc
    })

    print("F1:", f1)


Training: LogisticRegression
F1: 0.46974263853466264

Training: RandomForest
F1: 0.6646016176217519

Training: ExtraTrees
F1: 0.6452446906740535

Training: GradientBoosting
F1: 0.5949269792467333

Training: HistGradientBoosting
F1: 0.6475495307612096


In [48]:
results_df = pd.DataFrame(results).sort_values(by="F1", ascending=False)

print("\nModel Comparison:")
display(results_df)


Model Comparison:


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
1,RandomForest,0.699368,0.637925,0.693606,0.664602,0.771373
4,HistGradientBoosting,0.687182,0.627273,0.669181,0.647550,0.761691
2,ExtraTrees,0.703687,0.664006,0.627514,0.645245,0.775621
3,GradientBoosting,0.674842,0.639669,0.556034,0.594927,0.743822
0,LogisticRegression,0.647231,0.662525,0.363865,0.469743,0.717684


In [50]:
# Select best model
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Best model:", best_model_name)

Best model: RandomForest


In [51]:
# Detailed evaluation
y_pred = best_model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

         0.0       0.75      0.70      0.73      3699
         1.0       0.64      0.69      0.66      2784

    accuracy                           0.70      6483
   macro avg       0.70      0.70      0.70      6483
weighted avg       0.70      0.70      0.70      6483


Confusion Matrix:
[[2603 1096]
 [ 853 1931]]


In [52]:
# Save best model
MODEL_PATH = r"C:\Users\DELL\Desktop\astro\models\best_classification_model.pkl"

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

joblib.dump(best_model, MODEL_PATH)

print("Model saved:", MODEL_PATH)

Model saved: C:\Users\DELL\Desktop\astro\models\best_classification_model.pkl
